# Demo 3: Custom scorers

**Concept**: built-in judges cover common criteria. Your product has *specific* ones. Our tutor should give beginners a **runnable code example**, and it should **stay in its role**. MLflow offers a spectrum:

| Level | Type | We use it for | `source` in feedback |
|---|---|---|---|
| Moderate | **Guideline judge**: natural-language rule | "includes a short code example" | `LLM_JUDGE` |
| Full (deterministic) | **`@scorer`**: plain Python | "response contains a code snippet" | `CODE` |
| Full (LLM) | **`make_judge()`**: custom LLM judge | "stays in the tutor role" | `LLM_JUDGE` |

All of them run together in **one** `mlflow.genai.evaluate()` call, next to the built-ins from demo 2.

In [0]:
%pip install -U "mlflow[databricks]>=3.1" openai databricks-sdk
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 107.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 74.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 123.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 89.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.6/771.6 kB 36.5 MB/s  0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.13
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-396c77ab-16a2-4a6f-bf76-f00df49e9e34
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: idna
    Found exi

In [0]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import mlflow

from common.config import EXPERIMENT_PATH

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_PATH)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1571690280311094', creation_time=1792292632487, effective_trace_archival_retention=None, experiment_id='1571690280311094', last_update_time=1792294498146, lifecycle_stage='active', name='/Shared/python-wiki-agent-experiment', tags={'mlflow.experiment.sourceName': '/Shared/python-wiki-agent-experiment',
 'mlflow.experimentKind': 'genai_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'pranav@platformatory.com',
 'mlflow.ownerId': '77162250982329'}, trace_location=None, workspace='default'>

In [0]:
from common.agent_traced import answer_question
from common.eval_data import EVAL_DATA


def predict_fn(question: str) -> dict:
    return answer_question(question)


## 1. Guideline judge (moderate customisation)

You write the rule in plain English, and MLflow turns it into an LLM judge. Domain experts can write these
without touching code.

**Tip**: refer to the input as **"the request"** and the output as **"the response"**. The judge extracts both from the trace.

Guidelines can be *global* (same rule for every row, as here) or *per-row* (a different rule per example, stored in the dataset's expectations).

In [0]:
from mlflow.genai.scorers import Guidelines

code_example_guideline = Guidelines(
    name="includes_code_example",
    guidelines=(
        "If the request asks how to do something in Python, the response must include a short Python code example."
    ),
)

## 2. Deterministic scorer (full customisation, plain code)

When a check can be written as code, write it as code: it is free, instant and repeatable. The `@scorer` decorator turns a function into a scorer. It may take any of `inputs`, `outputs`, `expectations`, `trace`.

It returns a `Feedback` (value + **rationale**), and MLflow sets `source` to `CODE` for us.

Two details kept on purpose, because demo 6 registers scorers for production monitoring: imports are **inside** the function, and there are **no type hints** that need imports.

In [0]:
from mlflow.genai.scorers import scorer


@scorer
def has_code_snippet(outputs):
    from mlflow.entities import Feedback

    text = outputs["response"] if isinstance(outputs, dict) else str(outputs)
    found = "```" in text or text.count("`") >= 2
    return Feedback(
        value=found,
        rationale="Found a code block or inline code." if found else "No code snippet in the response.",
    )


Notice that this overlaps with the guideline judge on purpose. Two ways to check *similar* things. One is an LLM that understands intent, the other is a cheap string check. Where they **disagree** is interesting.

## 3. Custom LLM judge with `make_judge()` (full customisation)

For criteria that need judgement but do not fit a one-line guideline: **does the tutor stay in role?** (answers Python questions, declines off-topic ones, refuses to reveal its prompt). You describe the criteria in natural language, with `{{ inputs }}` and `{{ outputs }}` as placeholders, and MLflow handles the LLM call, parsing and the `Feedback` object. Instructions live in `common/judges.py`, since demo 4 reuses them.

This is **version 1**: a quick, vague first attempt, like most first attempts.

In [0]:
from common.judges import ROLE_INSTRUCTIONS_V1, make_role_judge

print(ROLE_INSTRUCTIONS_V1)
role_judge_v1 = make_role_judge(version=1)

Check whether the response stays in the role of a friendly Python tutor for beginners. The request is in {{ inputs }} and the response is in {{ outputs }}. Answer true if the response stays in role, false if it does not.


## 4. Run everything in one call

Built-ins from demo 2 plus the three custom scorers. The built-ins are kept so the results are comparable.

In [0]:
from mlflow.genai.scorers import Correctness, RetrievalGroundedness

results = mlflow.genai.evaluate(
    data=EVAL_DATA,
    predict_fn=predict_fn,
    scorers=[
        Correctness(),
        RetrievalGroundedness(),
        code_example_guideline,  # LLM_JUDGE, guideline
        has_code_snippet,  # CODE
        role_judge_v1,  # LLM_JUDGE, custom
    ],
)

print(f"run_id: {results.run_id}")
results.metrics

2026/10/18 04:00:12 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/18 04:00:12 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?]

run_id: e59d89177f8d44428025bc6d767a5e6f


{'includes_code_example/mean': np.float64(1.0),
 'correctness/mean': np.float64(0.9),
 'retrieval_groundedness/mean': np.float64(0.4),
 'has_code_snippet/mean': np.float64(0.9),
 'tutor_role/mean': np.float64(0.9)}

## 5. What to look at

In the **Evaluations** tab of the experiment:

1. **`source`**: the code-based scorer shows `CODE`, the other two show `LLM_JUDGE`.
2. **`includes_code_example` vs `has_code_snippet`**: filter for rows where they disagree. Read both rationales. Which one is right in each case? What does each get wrong?
3. **`tutor_role`** on the out-of-scope row (Java generics) and the adversarial row. Read the rationale, and open the trace to read the actual response.

## 6. Break it

An LLM judge is just another LLM: it can be wrong, vague, or biased. We wrote `tutor_role` v1 in one minute. **Pick three rows, read the response yourself, and decide: do you agree with the judge?**

If an evaluation relies on a judge you have never checked, the scores may be wrong in a way that looks authoritative.

**Next: demo 4 checks the judge against human opinions and improves it.**